In [1]:
import os
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

from src.waveform.utils import load_waveform, trim_waveform

In [72]:
from src.waveform.features_extraction.all_features import *

def waveform_all_features(waveform, sr):    
    mean_value = waveform_mean_value(waveform)
    variance = waveform_variance(waveform)
    std_dev = waveform_std_dev(waveform)
    skewness = waveform_skewness(waveform)
    kurt = waveform_kurtosis(waveform)
    slope_sign_changes = waveform_slope_sign_changes(waveform)
    mean_absolute_value = waveform_mean_absolute_value(waveform)
    logarithm_detector = waveform_logarithm_detector(waveform)
    average_amplitude_change = waveform_average_amplitude_change(waveform)
    difference_absolute_deviation = waveform_difference_absolute_deviation(waveform)
    integrated_absolute_value = waveform_integrated_absolute_value(waveform)
    mean_logarithm_kernel = waveform_mean_logarithm_kernel(waveform)
    simple_square_integral = waveform_simple_square_integral(waveform)
    third_moment = waveform_moments(waveform, 3)
    fourth_moment = waveform_moments(waveform, 4)
    fifth_moment = waveform_moments(waveform, 5)
    maximum_amplitude = waveform_maximum_amplitude(waveform)
    power_spectrum_ratio = waveform_power_spectrum_ratio(waveform)
    peak_frequency = waveform_peak_frequency(waveform)
    mean_power = waveform_mean_power(waveform)
    total_power = waveform_total_power(waveform)
    variance_of_central_frequency = waveform_variance_of_central_frequency(waveform, sr)
    activity, mobility, complexity = waveform_hjorth_parameters(waveform)

    waveform_l = waveform_length(waveform)
    glottal_features = waveform_glottal_features(waveform, sr)
    tempo_spectral_features = waveform_tempo_spectral_features(waveform, sr)
    formant_features = waveform_formant_features(waveform)
    other_features = waveform_other_features(waveform, sr)
    
    shimmer = waveform_shimmer(waveform)
    jitter = waveform_jitter(waveform)
    hnr = waveform_hnr(waveform)
    harmonicity = waveform_harmonicity(waveform)
    voiced_unvoiced_ratio = waveform_voiced_unvoiced_ratio(waveform)
    
    
    all_features = {
        "mean_value": mean_value,
        "variance": variance,
        "std_dev": std_dev,
        "skewness": skewness,
        "kurtosis": kurt,
        "slope_sign_changes": slope_sign_changes,
        "mean_absolute_value": mean_absolute_value,
        "logarithm_detector": logarithm_detector,
        "average_amplitude_change": average_amplitude_change,
        "difference_absolute_deviation": difference_absolute_deviation,
        "integrated_absolute_value": integrated_absolute_value,
        "mean_logarithm_kernel": mean_logarithm_kernel,
        "simple_square_integral": simple_square_integral,
        "third_moment": third_moment,
        "fourth_moment": fourth_moment,
        "fifth_moment": fifth_moment,
        "maximum_amplitude": maximum_amplitude,
        "power_spectrum_ratio": power_spectrum_ratio,
        "peak_frequency": peak_frequency,
        "mean_power": mean_power,
        "total_power": total_power,
        "variance_of_central_frequency": variance_of_central_frequency,
        "hjorth_activity": activity,
        "hjorth_mobility": mobility,
        "hjorth_complexity": complexity,
        "waveform_length": waveform_l,
        "glottal_features": glottal_features,
        "tempo_spectral_features": tempo_spectral_features,
        "formant_features": formant_features,
        "other_features": other_features,
        "shimmer": shimmer,
        "jitter": jitter,
        "hnr": hnr,
        "harmonicity": harmonicity,
        "voiced_unvoiced_ratio": voiced_unvoiced_ratio
    }
    
    return all_features


In [86]:
def unwrap_dict(dictionary: dict):
    result = {f"{k}_{i}": val for k, v in dictionary.items() if isinstance(v, tuple) for i, val in enumerate(v)}
    result.update({k: v for k, v in dictionary.items() if not isinstance(v, tuple)})
    return result

In [87]:
fe = waveform_all_features(w, sr)
unwrap_dict(fe)

{'glottal_features_0': 0.0005368759152480321,
 'glottal_features_1': 0.000375,
 'glottal_features_2': 0.00045793357155845995,
 'tempo_spectral_features_0': 133.92857142857142,
 'tempo_spectral_features_1': 1047.5736254848016,
 'tempo_spectral_features_2': 1012.1869134921862,
 'tempo_spectral_features_3': 2154.5948995231606,
 'tempo_spectral_features_4': 0.029237268,
 'formant_features_0': 0.5217016,
 'formant_features_1': 0.5217016,
 'formant_features_2': 0.2303399,
 'other_features_0': 33.97701,
 'other_features_1': 279.5926,
 'other_features_2': 0.042970665,
 'other_features_3': 0.15589676111205722,
 'other_features_4': 20.122902021996637,
 'mean_value': -2.9711191e-05,
 'variance': 0.0016295244,
 'std_dev': 0.04036737,
 'skewness': 2.067386538866217,
 'kurtosis': 18.612374495844822,
 'slope_sign_changes': 176955.0,
 'mean_absolute_value': 0.019078393,
 'logarithm_detector': 0.018336495,
 'average_amplitude_change': 0.008506143,
 'difference_absolute_deviation': 0.008506143,
 'integr

In [27]:
PROJECT_DIR = Path(os.environ['PROJECT_DIR'])
PREPROCESSED_DATA_PATH = PROJECT_DIR / 'data/preprocessed_data'

In [28]:
df = pd.read_csv(PREPROCESSED_DATA_PATH / 'data.csv')
split_df = pd.read_csv(PREPROCESSED_DATA_PATH / 'split.csv').set_index('participant_id')
df['split'] = df['participant_id'].apply(lambda i: split_df.loc[i]['split'])

In [29]:
train_df = df[df['split'] == 'train']
test_df = df[df['split'] == 'test']

In [38]:
y_train = train_df['phq_binary'].to_numpy().astype(int)

In [ ]:
from tqdm.auto import tqdm
tqdm.pandas()

def calc(group):
    source = group.iloc[0]['source']
    waveform, sr = load_waveform(source)
    return group.progress_apply(
        lambda row: pd.Series(
            unwrap_dict(
                waveform_all_features(
                    waveform=trim_waveform(
                        waveform=waveform, 
                        sample_rate=sr, 
                        start_time=row['start_time'],
                        end_time=row['end_time']
                    ), 
                    sr=sr
                )
            )
        ), 
        axis=1
    )
    
X_train = train_df.groupby('participant_id').progress_apply(lambda group: calc(group), include_groups=False).reset_index(drop=True)

  0%|          | 0/197 [00:00<?, ?it/s]

  0%|          | 0/16 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/16 [00:00<?, ?it/s]

  0%|          | 0/7 [00:00<?, ?it/s]

...

  0%|          | 0/16 [00:00<?, ?it/s]

  0%|          | 0/7 [00:00<?, ?it/s]

  0%|          | 0/5 [00:00<?, ?it/s]

  0%|          | 0/2 [00:00<?, ?it/s]

  0%|          | 0/3 [00:00<?, ?it/s]

In [ ]:
import pickle
from time import time

with open(f'train-features-{int(time)}.pkl', 'wb') as f:
    pickle.dump(X_train, f)